# GenAI Intensive 4일차 실습 노트북: 실습 미션 1, 2, 3
**실습 미션 1, 2 는 이 노트북에서, 실습 미션 3 은 Claude Code 에서 합니다. 미션마다 기본 실행 → 바꿔 보기 → Q 답 적기 순서로 진행합니다.**

예제는 전공정 SL-FAB 문서 252건을 썼습니다. 미션은 같은 회사의 **후공정 조립 라인 SL-PKG 문서 160파일(문서번호 159개)** 을 씁니다. 두 코퍼스에는 겹치는 문서가 한 건도 없습니다.
미션 1, 2 의 셀 코드는 모두 채워져 있습니다. 먼저 그대로 실행해 기본 결과를 보고, 바꿔 보기 셀의 ✏️ 줄 사이 값을 안내대로 고쳐 실행한 뒤, 결과가 어떻게 달라졌는지 미션 끝의 **Q** 답 셀에 적습니다.
**확장 미션(심화)** 은 기본 미션을 마친 경우에만 합니다. 목적만 받고 어떻게 만들지 스스로 정해, PC 의 Claude Code 로 직접 만들어 봅니다. 만들어 본 소감은 미션 끝의 「회고」 아래 답 셀에 적습니다.
미션 셀에 적힌 예제 번호(예제 1-2 등)는 `day04-examples.ipynb` 와 강의안 코드 장의 번호입니다.

**✏️ 표시 보는 법**: 실습 미션 도중 고쳐야 하는 칸에는 ✏️ 가 붙어 있습니다.
- **✏️ 직접 고칠 곳**: 꼭 고쳐야 하는 칸입니다. Q 의 답이 이 칸을 고쳐 실행한 결과에서 나옵니다(실습 미션 1, 2 의 바꿔 보기 셀).
- **✏️ 더 해 보기(선택)**: 기본 미션을 마친 뒤 해 보는 칸입니다. 지금 들어 있는 값으로 먼저 실행하고, 값을 바꿔 다시 실행하면 결과가 어떻게 달라지는지 볼 수 있습니다(실습 미션 2 (a) 셀의 내 질문).
- **✏️ 답 적는 곳**: Q 와 확장 미션 회고 아래의 답 셀입니다.

코드 안에서는 `# ✏️ ========` 두 줄 사이가 손대는 칸입니다.

**처음 한 번만 할 일: 코랩**
1. [파일] → [Drive에 사본 저장] 을 선택하고, 새 탭에 열린 사본에서 진행합니다.
2. 왼쪽 열쇠 아이콘(보안 비밀)에 2일차에 넣은 `ANTHROPIC_API_KEY` 가 있는지 보고, 그 줄의 「노트북 액세스」를 켭니다. 없으면 「새 보안 비밀 추가」에서 이름 `ANTHROPIC_API_KEY`, 값에 받은 키를 붙여 넣습니다.
3. 아래 「환경 준비」 셀을 위에서부터 차례로 실행합니다(Shift + Enter). (1) 미션 재료 받기와 설치, (2) API 키와 연결 확인 순서입니다. (2) 셀에 `연결 확인:` 뒤로 수도 이름이 나오면 됩니다.
4. 「미션 1, 2 준비」 셀부터 실행합니다. 준비 셀은 예제 셀을 폴더 이름만 바꿔 옮긴 것이라, 예제 노트북 없이 이 노트북만으로 실행됩니다.
5. 실습 미션 3 은 이 노트북이 아니라 Claude 앱 Code 탭에서 `day04-wiki/` 폴더로 합니다. 노트북 맨 아래 안내를 봐 주세요.

## 환경 준비

**환경 준비 (1) 미션 재료 받기와 설치**

미션 재료 zip(문서 폴더 `mission_corpus/`, 문서 목록표 `mission_manifest.csv`, 평가셋 `mission_evalset.csv`, 채점 프롬프트 `judge_prompt.txt`, 재료 `4일차_미션1_재료.txt`, `4일차_미션2_재료.txt`, `4일차_미션3_재료.txt`)을 내려받아 풀고 라이브러리 세 개를 설치합니다. `mission_corpus/` 폴더가 이미 있으면 내려받기를 건너뜁니다.

**출력에서 볼 것**: `미션 문서 파일 160 개` 가 보이면 됩니다.

In [ ]:
# 환경 준비 (1) 미션 재료 받기와 설치
import os, zipfile, urllib.request
URL = 'https://lecture-web-five.vercel.app/materials/day04v2-missions.zip'
# 재료가 이미 있으면 다시 받지 않습니다
if not os.path.exists('mission_corpus'):
    urllib.request.urlretrieve(URL, 'day04v2-missions.zip')
    zipfile.ZipFile('day04v2-missions.zip').extractall('.')
%pip -q install anthropic rank-bm25 sentence-transformers
print('미션 문서 파일', len(os.listdir('mission_corpus')), '개')

**환경 준비 (2) API 키와 Claude 연결 확인**

코랩 「보안 비밀」에 넣은 `ANTHROPIC_API_KEY` 를 읽고, 없으면 키 입력 칸을 표시합니다. 이어서 이 노트북 전체가 쓰는 `llm(prompt)` 함수를 정의합니다. `llm` 은 Claude 에 질문 하나를 보내 답 글자만 반환하고, 같은 질문을 다시 보내면 `_MEMO` 에 저장해 둔 답을 그대로 반환합니다.

**출력에서 볼 것**: `연결 확인:` 뒤에 수도 이름이 나오면 키와 모델이 정상입니다. 키 오류가 나오면 보안 비밀 이름과 「노트북 액세스」를 다시 확인해 주세요.

In [ ]:
# 환경 준비 (2) API 키와 모델: 2일차와 같은 방식입니다
# 코랩 왼쪽 열쇠 모양 「보안 비밀」에 ANTHROPIC_API_KEY 를 넣고 「노트북 액세스」를 켭니다.
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY').strip()
# 코랩이 아니거나 보안 비밀이 없으면 넘어갑니다
except Exception:
    pass
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        import getpass
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
    except Exception:
        print('[안내] 키가 없습니다 · 보안 비밀 이름 ANTHROPIC_API_KEY 를 확인합니다')

import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5-5'                # 2일차 기본 모델과 같습니다

def text_of(resp):
    """응답에서 text 블록만 이어 붙입니다. thinking 블록은 건너뜁니다."""
    return ''.join(b.text for b in resp.content if b.type == 'text').strip()

# 같은 셀을 다시 실행해도 같은 요청은 두 번 보내지 않습니다
_MEMO = {}
def llm(prompt, max_tokens=2000):
    """질문 글을 받아 Claude 의 답 글자만 돌려줍니다 · max_tokens 2000 은 긴 답도 잘리지 않게 둔 상한입니다."""
    if prompt in _MEMO:
        return _MEMO[prompt]
    resp = client.messages.create(
        model=MODEL, max_tokens=max_tokens,
        # 추론 수준(effort)을 낮게 정해 요청합니다
        output_config={'effort': 'low'},
        messages=[{'role': 'user', 'content': prompt}])
    out = text_of(resp)
    if not out:                            # 빈 답은 저장하지 않고 이유를 보여 줍니다
        return f'[빈 응답] stop_reason={resp.stop_reason} · max_tokens 를 늘립니다'
    _MEMO[prompt] = out
    return out

print('연결 확인:', llm('한 단어로만 답하세요. 대한민국의 수도는?'))

## 실습 미션 1: 검색기 네 개를 같은 평가셋으로 측정하고, 설정 값을 바꿔 비교합니다
**검색기 네 개를 평가셋 10문항으로 측정하고, 설정 값을 하나씩 바꿔 결과가 어떻게 달라지는지 봅니다.**

재료는 `4일차_미션1_재료.txt` 와 미션 코퍼스(`mission_corpus/`), 평가셋 `mission_evalset.csv` 10문항입니다. 예제 파트 1 의 검색기 네 개(벡터, 키워드, 하이브리드, 리랭킹)를 처음 보는 문서로 측정합니다. 결과표의 칸은 「hit@1 / hit@3」 입니다. hit@1 은 정답 문서가 1위에 든 문항의 비율(%), hit@3 은 3위 안에 든 문항의 비율(%)입니다.

**✏️ 이 미션에서 손댈 곳**
- 직접 고칠 곳: 실습 미션 1 바꿔 보기 1 셀의 `KEYWORD_TITLE`, 바꿔 보기 2-1 셀의 `KEYWORD_WEIGHT`, 바꿔 보기 2-2 셀의 `RERANK_TOP`, 바꿔 보기 3 셀의 `SEARCHER`
- 답 적는 곳: Q1, Q2, Q3
- 「미션 1, 2 준비」, 「미션 1 실행 준비」 셀과 실습 미션 1 (a), (b) 셀은 실행만 합니다.

**하는 순서**
1. 「미션 1, 2 준비」, 「미션 1 실행 준비」 셀과 실습 미션 1 (a), (b) 셀을 실행해 기본 결과를 봅니다. (a) 가 결과표 첫 줄(기준 줄)을 만듭니다.
2. **바꿔 보기 1, 2, 3**: 바꿔 보기 셀마다 ✏️ 줄 사이의 값 하나를 안내대로 고쳐 그 셀을 실행합니다. 바꿔 보기 셀은 그 셀의 값 하나만 기준 설정과 다르게 써서 측정하고, 결과표에 한 줄을 더해 기준 줄과 비교합니다. 다른 바꿔 보기 셀에서 고친 값은 쓰지 않으므로 결과가 서로 섞이지 않습니다.
3. 달라진 결과를 보고 미션 1 끝의 **Q1, Q2, Q3** 답 셀에 적습니다.

미션 1 은 Claude 를 부르지 않아 다시 실행해도 숫자가 같습니다.

### 미션 1, 2 준비: 미션 코퍼스 색인
**미션 1, 2 가 함께 쓰는 색인과 측정 도구를 만드는 셀입니다. 실행만 합니다.**

미션 코퍼스를 절 단위로 잘라 벡터 색인을 만들고, 미션 1 에서 검색기를 측정할 도구를 정의합니다. 예제 노트북 환경 준비 (3) 의 색인 코드, 예제 1-1 (2), (4) 의 검색, 측정 함수, 예제 1-2 (1) 의 `tok()` 을 한 셀로 합쳤고, 폴더 이름만 다릅니다.

- 청크 만들기(맨 위 `for` 문): `mission_corpus/*.md` 를 `## ` 절 단위로 자릅니다. 절이 있는 문서의 머리 청크는 앞 8줄만 두고, 청크마다 본문은 1200자까지 둡니다.
- `TITLE` 은 문서 제목 사전, `E` 는 「문서 제목 · 절 제목」을 붙인 청크 임베딩, `DOCT` 는 문서번호별 청크 목록입니다.
- `vec_rank(q)`: 벡터 점수가 높은 청크 번호 50개를 반환합니다.
- `to_docs(idx, k)`: 청크 순위를 문서 순위로 바꿉니다. 같은 문서의 청크는 처음 나온 자리 하나만 집계합니다.
- `vector_search(q)`: 위 두 함수를 이어 문서 10건을 반환하는 벡터 검색기입니다.
- `metrics(search_fn, evset)`: 정답 문서가 1위 안(hit@1), 3위 안(hit@3)에 든 문항의 비율과 MRR(정답 등수 역수의 평균)을 계산합니다. 평가셋의 정답 문서 칸(`gold`)이 비어 있는 문항은 뺍니다.
- `show(name, m)`: 그 결과를 한 줄로 출력합니다.
- `tok(s)`: 글을 단어와 한글 두 글자 조각으로 나눕니다. 예제 1-2 (1) 의 키워드 검색과 같은 함수이고, 미션 1 실행 준비 셀의 키워드 검색 색인이 씁니다.

**출력에서 볼 것**: `청크 541 · 문서 159 · 평가셋 10 문항` 이 보이면 됩니다.

In [ ]:
# 미션 1, 2 준비: 미션 코퍼스를 절 단위로 잘라 벡터 색인을 만들고 검색기를 측정할 도구를 정의합니다
import re, csv, json, time, unicodedata
import numpy as np
from collections import namedtuple, defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
from sentence_transformers import SentenceTransformer

# 예제 노트북 환경 준비 (3) 과 같은 임베딩 모델입니다
model = SentenceTransformer('intfloat/multilingual-e5-small')
META = {r['file']: r for r in csv.DictReader(open('mission_manifest.csv', encoding='utf-8-sig'))}
Chunk = namedtuple('Chunk', 'doc_id version type date section text')
chunks = []
for raw in sorted(os.listdir('mission_corpus')):
    # 목록표 조회용 이름. 파일은 원래 이름 raw 로 엽니다
    fn = unicodedata.normalize('NFC', raw)
    if not fn.endswith('.md'):
        continue
    meta = META[fn]
    # '## ' 로 시작하는 줄 앞에서 잘라 문서 머리(head)와 절 목록(secs)으로 나눕니다
    head, *secs = re.split(r'\n(?=## )', open(f'mission_corpus/{raw}', encoding='utf-8').read())
    for s in [head] + secs:
        s = s.strip()
        # 절이 있는 문서의 머리 청크는 앞 8줄만 둡니다
        if secs and s == head.strip():
            s = '\n'.join(s.splitlines()[:8])
        chunks.append(Chunk(meta['doc_id'], meta['version'], meta['type'], meta['date'],
                            # 절 제목(첫 줄)과 본문 1200자까지
                            s.splitlines()[0].lstrip('# ').strip(), s[:1200]))
TITLE = {}
for c in chunks:
    # 문서 첫 청크의 절 제목을 문서 제목으로 씁니다
    TITLE.setdefault((c.doc_id, c.version), c.section)
E = model.encode([f'passage: {TITLE[c.doc_id, c.version]} · {c.section}\n{c.text}' for c in chunks],
                 normalize_embeddings=True, batch_size=64)
DOCT = defaultdict(list)
for c in chunks:
    DOCT[c.doc_id].append(c)

def vec_rank(q, n=50):
    """질문 글을 받아 벡터 점수가 높은 청크 번호 n개(기본 50)를 돌려줍니다."""
    qe = model.encode(['query: ' + q], normalize_embeddings=True)[0]
    return list(np.argsort(-(E @ qe), kind='stable')[:n])

def to_docs(idx, k=10):
    """청크 번호 순위를 받아 중복 없는 문서번호 순위 k개(기본 10)를 돌려줍니다."""
    seen = []
    for i in idx:
        if chunks[i].doc_id not in seen:
            seen.append(chunks[i].doc_id)
        if len(seen) >= k:
            break
    return seen

def vector_search(q):
    """질문 글을 받아 벡터 검색으로 문서번호 10건을 돌려줍니다."""
    return to_docs(vec_rank(q), 10)

def metrics(search_fn, evset):
    """검색 함수와 평가셋을 받아 hit@1 · hit@3 (%) · MRR · 문항 수를 사전으로 돌려줍니다."""
    h1 = h3 = 0
    mrr = 0.0
    n = 0
    for e in evset:
        if not e['gold']:
            continue
        ranked = search_fn(e['q'])
        n += 1
        # 정답 문서의 등수(0부터), 없으면 None 입니다
        rank = next((i for i, doc in enumerate(ranked) if doc == e['gold']), None)
        if rank is not None:
            mrr += 1 / (rank + 1)
            h1 += rank < 1
            h3 += rank < 3
    return dict(hit1=round(h1 / n * 100, 1), hit3=round(h3 / n * 100, 1), mrr=round(mrr / n, 3), n=n)

def show(name, m):
    """검색기 이름과 metrics 결과 사전을 받아 한 줄로 출력합니다."""
    print(f"{name:6s} hit@1 {m['hit1']:5.1f}   hit@3 {m['hit3']:5.1f}   MRR {m['mrr']:.3f}")

# 키워드 검색이 쓰는 토큰 나누기. 예제 1-2 (1) 의 tok() 와 같습니다 (미션 1 실행 준비 셀이 씁니다)
def tok(s):
    """글을 받아 단어와 한글 두 글자 조각의 목록을 돌려줍니다."""
    s = re.sub(r'[^\w가-힣]+', ' ', s.lower())
    out = []
    for w in s.split():
        out.append(w)
        if len(w) > 2 and re.match(r'^[가-힣]+$', w):
            out += [w[i:i + 2] for i in range(len(w) - 1)]
    return out

EV = list(csv.DictReader(open('mission_evalset.csv', encoding='utf-8-sig')))
print('청크', len(chunks), '· 문서', len(DOCT), '· 평가셋', len(EV), '문항')

**미션 1 실행 준비**: 실행만 합니다

이 셀은 도구를 만드는 셀이라 코드를 읽지 않아도 됩니다. 실행한 뒤 아래 표의 설정 세 개만 기억해 두세요.

네 검색기(벡터, 키워드, 하이브리드, 리랭킹)를 만드는 함수와, 결과표에 한 줄씩 더하는 `run1` 을 정의합니다. 처음 실행할 때 리랭커 모델을 내려받습니다. 결과표는 다음 (a) 셀이 만듭니다.

| 설정 | 뜻 | 기준 설정 값 |
|---|---|---|
| 키워드에 제목 (`KEYWORD_TITLE`) | 키워드 검색 색인에 청크마다 문서 제목을 붙일지 | True |
| 키워드 가중 (`KEYWORD_WEIGHT`) | 하이브리드에서 키워드 순위를 벡터 순위의 몇 배로 칠지 (0 이면 벡터만) | 1 |
| 리랭커 후보 (`RERANK_TOP`) | 하이브리드 순위 앞 몇 개를 리랭커가 다시 매길지 | 30 |

**출력에서 볼 것**: `미션 1 실행 준비 완료` 가 보이면 됩니다.

In [ ]:
# 미션 1 실행 준비: 검색기 네 개를 만드는 함수와 결과표에 한 줄을 더하는 run1 을 정의합니다. 고치지 않고 실행합니다
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import pandas as pd

# 기준 설정: (a) 셀과, 바꿔 보기 셀이 고치지 않는 값은 이 값을 씁니다
BASE = dict(title=True, weight=1, top=30)
# 설정 키를 결과표에 보일 이름입니다
NAME1 = {'title': '키워드에 제목', 'weight': '키워드 가중', 'top': '리랭커 후보'}
_BM = {}
def bm_rank(q, title=True, n=50):
    """질문 글을 받아 BM25 점수가 높은 청크 번호 n개(기본 50)를 돌려줍니다 · title 은 색인에 문서 제목을 붙일지입니다."""
    # 제목을 넣은 색인, 뺀 색인을 한 번씩만 만듭니다
    if title not in _BM:
        _BM[title] = BM25Okapi([tok((TITLE[c.doc_id, c.version] + ' ' if title else '') + c.text) for c in chunks])
    return list(np.argsort(-_BM[title].get_scores(tok(q)), kind='stable')[:n])

# 예제 1-3 의 RRF: 목록마다 가중을 곱합니다. k=60 은 예제와 같은 값입니다
def rrf(lists, weights, k=60):
    """순위 목록들과 가중 목록을 받아 RRF 점수가 높은 순으로 청크 번호를 돌려줍니다."""
    sc = defaultdict(float)
    for L, w in zip(lists, weights):
        for r, i in enumerate(L):
            sc[i] += w / (k + r + 1)
    return sorted(sc, key=sc.get, reverse=True)

def hyb_rank(q, title=True, weight=1):
    """질문 글을 받아 벡터 순위와 키워드 순위를 RRF 로 합친 청크 번호 순위를 돌려줍니다."""
    return rrf([vec_rank(q), bm_rank(q, title)], [1, weight])

# 예제 1-4 와 같은 리랭커 모델: 질문+청크를 512 토큰까지 봅니다
rr = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512)
def rerank_rank(q, title=True, weight=1, top=30):
    """질문 글을 받아 하이브리드 순위 앞 top 개를 리랭커 점수로 다시 매긴 청크 번호 순위를 돌려줍니다."""
    # 하이브리드 순위 앞 top 개만 리랭커가 다시 매깁니다 (예제 1-4)
    cand = hyb_rank(q, title, weight)[:top]
    # 제목 + 본문 앞 400자만 리랭커에 넣습니다
    sc = rr.predict([[q, TITLE[chunks[i].doc_id, chunks[i].version] + ' ' + chunks[i].text[:400]] for i in cand],
                    show_progress_bar=False)
    return [cand[j] for j in np.argsort(-sc, kind='stable')]

def make_searchers(title, weight, top):
    """설정 값 세 개를 받아 검색기 네 개(벡터 · 키워드 · 하이브리드 · 리랭킹)를 이름별 사전으로 돌려줍니다."""
    def keyword(q): return to_docs(bm_rank(q, title), 10)
    def hybrid(q): return to_docs(hyb_rank(q, title, weight), 10)
    def rerank(q): return to_docs(rerank_rank(q, title, weight, top), 10)
    return {'벡터': vector_search, '키워드': keyword, '하이브리드': hybrid, '리랭킹': rerank}

LOG1 = []      # 결과표 줄: run1 이 실행마다 하나씩 더합니다
RANKS1 = []    # 실행마다 검색기별, 문항별 정답 문서 등수
def run1(title=True, weight=1, top=30):
    """세 값으로 네 검색기를 측정해 결과표에 한 줄을 더하고, 기준 줄과 비교해 등수가 바뀐 문항을 보입니다."""
    s = dict(title=title, weight=weight, top=top)
    for key, v in s.items():
        # 키워드 가중만 0.5 같은 소수도 받습니다
        if type(v) is not type(BASE[key]) and not (key == 'weight' and type(v) is float):
            return print(f'[안내] {NAME1[key]} 에 {v!r} 는 넣을 수 없습니다 · 처음 값처럼 따옴표 없이 넣어 주세요 (예: False · 2 · 0.5 · 10)')
    changed = [key for key in s if s[key] != BASE[key]]
    # 기준 줄이 먼저 있어야 비교할 수 있어 기준부터 측정합니다
    if not LOG1 and changed:
        run1()
    if LOG1 and not changed:
        print('바꿀 값이 기준 설정과 같아 줄을 더하지 않았습니다 · 지금까지의 결과표입니다')
        return display(pd.DataFrame(LOG1).set_index('실행'))
    if len(changed) > 1:
        print('[안내] 값 두 개 이상을 함께 바꿨습니다 · 이 줄만으로는 어느 값 때문인지 구분할 수 없습니다')
    S = make_searchers(**s)
    label = '기준' if not changed else ' · '.join(f'{NAME1[key]} {s[key]}' for key in changed)
    same = sum(r['실행'].split(' (')[0] == label for r in LOG1)
    # 같은 설정을 다시 측정하면 몇 번째인지 붙입니다
    if same:
        label += f' ({same + 1}번째)'
    row = {'실행': label, '키워드에 제목': s['title'], '키워드 가중': s['weight'], '리랭커 후보': s['top']}
    ranks = {}
    for nm, f in S.items():
        m = metrics(f, EV)
        row[nm] = f"{m['hit1']:.0f} / {m['hit3']:.0f}"
        ranks[nm] = {}
        for e in EV:
            found = f(e['q'])
            ranks[nm][e['gold']] = found.index(e['gold']) + 1 if e['gold'] in found else '밖'
    LOG1.append(row)
    RANKS1.append(ranks)
    print('결과표 · 칸마다 hit@1 / hit@3 (%) · 맨 위 줄이 기준입니다')
    display(pd.DataFrame(LOG1).set_index('실행'))
    if len(RANKS1) > 1:
        print('기준 줄과 비교해 정답 문서의 등수가 바뀐 문항 (밖 = 10위 밖)')
        moved = []
        for nm in S:
            for g, r in ranks[nm].items():
                if RANKS1[0][nm][g] != r:
                    moved.append((nm, g, RANKS1[0][nm][g], r))
        for nm, g, a, b in moved:
            print(f'   {nm:6s} {g:18s} {a}위 → {b}위'.replace('밖위', '밖'))
        if not moved:
            print('   없음 · 이 값은 평가셋 10문항의 등수를 바꾸지 않았습니다')

def doc_text(doc_id):
    """문서번호를 받아 그 문서의 청크 본문을 이어 붙인 글을 돌려줍니다."""
    return '\n'.join(c.text for c in DOCT[doc_id])

def compare(q, gold, top1):
    """정답 문서와 1위 문서에서 서로 다른 줄, 질문 단어가 각 문서에 있는지를 보입니다."""
    gold_lines = doc_text(gold).splitlines()
    top1_lines = doc_text(top1).splitlines()
    same = len(set(gold_lines) & set(top1_lines))
    print(f'   두 문서에 똑같이 있는 줄 {same}개 · 다른 줄만 보입니다')
    for tag, ln in [('정답', x) for x in gold_lines if x not in top1_lines][:6] + [('1위 ', x) for x in top1_lines if x not in gold_lines][:6]:
        print(f'     {tag} │ {ln[:70]}')
    qw = [w for w in dict.fromkeys(re.sub(r'[^\w가-힣]+', ' ', q).split()) if len(w) > 1]
    gold_body = doc_text(gold)
    top1_body = doc_text(top1)
    print('   질문 단어 중 정답 문서에 있는 것 ', [w for w in qw if w in gold_body])
    print('   질문 단어 중 1위 문서에 있는 것  ', [w for w in qw if w in top1_body])

print('미션 1 실행 준비 완료 · 다음 (a) 셀이 결과표 기준 줄을 만듭니다')

In [ ]:
# 실습 미션 1 (a) 네 검색기를 평가셋 10문항으로 측정합니다 (예제 1-2 키워드, 1-3 하이브리드, 1-4 리랭킹)
# 볼 것: 결과표 칸마다 hit@1 / hit@3 (%), 이 줄이 기준 줄입니다

# 기준 설정: 키워드에 제목 True, 키워드 가중 1, 리랭커 후보 30
run1(**BASE)


In [ ]:
# 실습 미션 1 (b) 리랭킹 검색기가 1위를 놓친 문항을 열어 정답 문서와 1위 문서를 비교합니다 (예제 1-1 (3), 1-5)
# 볼 것: 문항마다 정답 문서의 등수, 두 문서에서 서로 다른 줄, 질문 단어가 어느 문서에 있는지

def show_missed(searcher):
    """검색기 이름(문자열)을 받아, 기준 설정으로 만든 그 검색기가 1위를 놓친 문항을 모두 출력합니다 (반환값 없음)."""
    searchers = make_searchers(**BASE)
    if searcher not in searchers:
        return print(f"[안내] 검색기는 {' · '.join(repr(name) for name in searchers)} 가운데 하나를 따옴표까지 넣어 주세요")
    search = searchers[searcher]
    missed = 0
    for e in EV:
        ranked = search(e['q'])
        if ranked[0] != e['gold']:
            missed += 1
            gold_rank = ranked.index(e['gold']) + 1 if e['gold'] in ranked else '밖'
            print(f"\n질문 · {e['q']}\n   정답 {e['gold']} ({gold_rank}위)   ·   1위 {ranked[0]}")
            compare(e['q'], e['gold'], ranked[0])
    print(f'\n{searcher} 검색기가 1위를 놓친 문항 {missed} / {len(EV)}')

show_missed('리랭킹')


### 미션 1: 바꿔 보고 답하기
**바꿔 보기 셀마다 ✏️ 줄 사이의 값 하나를 안내대로 고쳐 실행합니다. 바꿔 보기 셀은 그 셀의 값 하나만 기준 설정과 다르게 쓰므로, 다른 바꿔 보기의 결과와 섞이지 않습니다.**

**✏️ 바꿔 보기 1** 실습 미션 1 바꿔 보기 1 셀에서 키워드 색인에 문서 제목을 붙일지 정하는 `KEYWORD_TITLE = True` 를 `False` 로 고쳐 실행합니다. 결과표의 키워드, 하이브리드 칸과 「등수가 바뀐 문항」을 봅니다.

**✏️ 바꿔 보기 2** 바꿔 보기 2-1 셀에서 하이브리드 검색이 키워드 순위에 주는 가중 `KEYWORD_WEIGHT = 1` 을 `0` 으로 고쳐 실행합니다(키워드 순위를 빼고 벡터 순위만 씁니다). 이어서 바꿔 보기 2-2 셀에서 리랭커가 순위를 다시 매길 후보 수 `RERANK_TOP = 30` 을 `10` 으로 고쳐 실행합니다. 두 줄의 리랭킹 칸과 SOP-BG-002 의 등수를 봅니다.

**✏️ 바꿔 보기 3** 바꿔 보기 3 셀에서 1위를 놓친 문항을 열어 볼 검색기 `SEARCHER = '리랭킹'` 을 `'하이브리드'` 로 고쳐 강사 화면과 함께 실행합니다. 실습 미션 1 (b) 에서 리랭킹이 1위를 놓친 문항과 비교해, 리랭킹이 해결한 문항과 둘 다 놓친 문항을 나눕니다.

In [ ]:
# 실습 미션 1 바꿔 보기 1: 키워드 색인에서 문서 제목을 빼고 측정합니다
# 볼 것: 결과표의 키워드, 하이브리드 칸, 기준 줄과 비교해 등수가 바뀐 문항

# ✏️ ======== 직접 고칠 곳: 키워드 색인에 문서 제목을 붙일지 ========
# 지금은 True 가 들어 있어 키워드 색인이 청크마다 문서 제목을 붙입니다.
# False 로 고쳐, 제목을 빼면 키워드 검색과 하이브리드 검색의 hit@1, hit@3 이 어떻게 바뀌는지 확인합니다.
KEYWORD_TITLE = True
# ✏️ ======== 여기까지 ========

# 나머지 두 값은 기준 설정 그대로
run1(title=KEYWORD_TITLE, weight=BASE['weight'], top=BASE['top'])


In [ ]:
# 실습 미션 1 바꿔 보기 2-1: 하이브리드에서 키워드 순위를 빼고 측정합니다
# 볼 것: 결과표의 리랭킹 칸, 기준 줄과 비교한 SOP-BG-002 의 등수

# ✏️ ======== 직접 고칠 곳: 하이브리드 검색에서 키워드 순위에 주는 가중 ========
# 지금은 1 이 들어 있어 하이브리드 검색이 키워드 순위와 벡터 순위를 같은 비중으로 합칩니다.
# 0 으로 고쳐, 키워드 순위를 빼고 벡터 순위만 쓰면 리랭킹 칸과 SOP-BG-002 의 등수가 어떻게 바뀌는지 확인합니다.
KEYWORD_WEIGHT = 1
# ✏️ ======== 여기까지 ========

# 나머지 두 값은 기준 설정 그대로
run1(title=BASE['title'], weight=KEYWORD_WEIGHT, top=BASE['top'])


In [ ]:
# 실습 미션 1 바꿔 보기 2-2: 리랭커가 다시 매길 후보 수를 줄여 측정합니다
# 볼 것: 결과표의 리랭킹 칸, 기준 줄과 비교한 SOP-BG-002 의 등수

# ✏️ ======== 직접 고칠 곳: 리랭커가 순위를 다시 매길 후보 수 ========
# 지금은 30 이 들어 있어 리랭커가 하이브리드 검색 상위 청크 30개의 순위를 다시 매깁니다.
# 10 으로 고쳐, 후보를 줄이면 리랭킹 칸과 SOP-BG-002 의 등수가 어떻게 바뀌는지 확인합니다.
RERANK_TOP = 30
# ✏️ ======== 여기까지 ========

# 나머지 두 값은 기준 설정 그대로
run1(title=BASE['title'], weight=BASE['weight'], top=RERANK_TOP)


In [ ]:
# 실습 미션 1 바꿔 보기 3: 다른 검색기가 1위를 놓친 문항을 (b) 와 같은 방식으로 엽니다
# 볼 것: (b) 의 리랭킹 실패 문항과 비교해, 리랭킹이 해결한 문항과 둘 다 놓친 문항

# ✏️ ======== 직접 고칠 곳: 1위를 놓친 문항을 열어 볼 검색기 ========
# 지금은 '리랭킹' 이 들어 있어 실습 미션 1 (b) 와 같은 문항이 열립니다.
# '하이브리드' 로 고쳐, 리랭킹이 해결한 문항과 두 검색기가 모두 놓친 문항을 확인합니다. 넣을 수 있는 값은 '벡터', '키워드', '하이브리드', '리랭킹' 입니다.
SEARCHER = '리랭킹'
# ✏️ ======== 여기까지 ========

show_missed(SEARCHER)


**Q1: 키워드 색인에서 문서 제목을 빼면 검색 점수가 어떻게 바뀌나요?**

- **의도**: 색인 설정 하나가 검색 점수를 얼마나 바꾸는지 숫자로 보고, 그 숫자로 설정을 고를 수 있는지 확인합니다.
- **입력**: (a), (b) 셀까지 실행한 뒤, 바꿔 보기 1 셀에서 `KEYWORD_TITLE = True` 를 `False` 로 고쳐 그 셀을 실행합니다.
- **기대 결과**: 결과표에 `키워드에 제목 False` 줄이 하나 더 생기고, 그 아래 「기준 줄과 비교해 정답 문서의 등수가 바뀐 문항」 목록이 나옵니다.
- **답에 적을 것**: (1) 키워드 칸의 hit@1, hit@3 이 기준 줄에서 얼마로 바뀌었는지 (2) 등수가 바뀐 문항 하나의 문서번호와 등수 (3) 상위 3개를 발췌로 붙이는 RAG 라면 제목을 빼는 설정을 쓸지와 그 이유 한 문장
- **답 예시(모양만)**: 키워드 hit@1 ○○ → ○○, hit@3 ○○ → ○○ / ○○○ 가 ○위 → ○위 / 쓰지 않습니다. ○○○ 가 상위 3개 밖으로 밀려 발췌에 들어오지 않기 때문입니다.

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

**Q2: 하이브리드 설정을 바꾸면 리랭킹 결과는 어떻게 되나요?**

- **의도**: 리랭커 모델은 그대로여도, 리랭커에 넘기는 후보가 바뀌면 리랭킹 결과가 달라진다는 것을 확인합니다.
- **입력**: 바꿔 보기 2-1 셀에서 `KEYWORD_WEIGHT = 1` 을 `0` 으로, 바꿔 보기 2-2 셀에서 `RERANK_TOP = 30` 을 `10` 으로 고쳐 각각 실행합니다.
- **기대 결과**: 결과표에 `키워드 가중 0` 줄과 `리랭커 후보 10` 줄이 생기고, 등수가 바뀐 문항 목록에 SOP-BG-002 가 나옵니다(`밖` 은 10위 밖이라는 뜻입니다).
- **답에 적을 것**: (1) 두 경우 각각 「등수가 바뀐 문항」 목록에서 SOP-BG-002 의 리랭킹 등수가 몇 위가 되었는지 (2) 리랭커 모델은 그대로인데 결과가 나빠진 까닭 한 문장 (3) 운영에서 리랭커 후보 수를 정하기 전에, 정답 문서가 하이브리드 순위 몇 위까지 오는지처럼 먼저 확인할 것 하나
- **답 예시(모양만)**: 키워드 가중 0: ○위(또는 밖), 리랭커 후보 10: ○위(또는 밖) / 정답 문서가 리랭커에 넘어가는 후보 안에 들지 못해서입니다 / ○○○ 를 먼저 확인합니다.

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

**Q3: 리랭킹도 1위를 놓친 문항은 왜 놓쳤나요?**

- **의도**: 검색기를 바꿔도 해결되지 않는 실패가 어떤 유형인지 나눠 보고, 검색기 말고 고칠 곳을 찾습니다.
- **입력**: (b) 셀 출력(리랭킹이 1위를 놓친 문항)을 읽습니다. 강사와 함께 실행한 바꿔 보기 3 셀 출력(하이브리드가 놓친 문항)과 비교합니다.
- **기대 결과**: 놓친 문항마다 정답 문서와 1위 문서의 등수, 두 문서에서 서로 다른 줄(「정답 │」 「1위 │」), 「질문 단어 중 …」 줄이 나옵니다.
- **답에 적을 것**: 놓친 문항마다 한 줄로 (1) 정답 문서번호 (2) 실패 유형 하나(본문이 같은 문서, 틀이 같고 값만 다른 문서, 한 절에 내용이 여럿, 청크 경계, 표현 불일치) (3) 그렇게 판단한 근거가 된 출력 줄. 마지막에, 하이브리드와 리랭킹이 둘 다 놓친 문항은 검색기 말고 무엇을 고칠지(문서, 청크, 질문, 평가셋 가운데) 한 문장
- **답 예시(모양만)**: ○○○, 틀이 같고 값만 다른 문서, 「1위 │」 줄과 「정답 │」 줄이 ○○ 값만 다름 / 둘 다 놓친 문항은 ○○ 를 고칩니다.

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 확장 미션 1 (심화): 검색기 비교 도구 만들기
**기본 미션을 마친 경우에만 합니다. 목적만 보고 어떻게 만들지 스스로 정한 뒤, Claude Code 로 직접 만들어 봅니다.**

**목적**
검색 방식을 바꿀 때마다 검색기 여러 개를 같은 평가셋으로 측정해 비교할 수 있는 도구가 필요합니다. 문서 폴더와 평가셋을 넣으면 검색기마다 정답 문서가 1위 안, 3위 안에 든 비율을 표로 보여 줘야 하고, 설정 값을 바꿔 다시 측정할 수 있어야 합니다.

**재료**: `mission_corpus/` 폴더, `mission_manifest.csv`, `mission_evalset.csv` 입니다. 학습 사이트 4일차의 `day04v2-missions.zip` 을 내 PC 에 받아 풀면 들어 있습니다. 코랩에 풀린 폴더는 Claude Code 가 열 수 없습니다. 본인 업무에 맞는 재료를 새로 만들어 진행해도 됩니다.

**만들기 전에 생각해 볼 것**
- 문서를 어떤 단위로 잘라 색인에 넣을까요?
- 검색기마다 결과가 다를 때, 어느 문항에서 왜 달랐는지 어떻게 확인할까요?
- 오늘 미션 1 에서 쓴 함수 가운데 무엇을 가져다 쓸 수 있을까요?

**진행**
1. 목적을 읽고, 어떤 단계로 만들지 먼저 짧게 메모합니다.
2. PC 에 작업 폴더를 만들고 재료 파일을 넣은 뒤, Claude Code 에서 그 폴더를 열어 **새 세션**을 시작합니다.
3. 목적과 내 계획을 Claude Code 에 보내 만들게 하고, 결과를 보며 고쳐 달라고 요청하면서 완성해 갑니다.
4. 다 만들면 바로 아래 「회고」에 직접 만들어 보니 어땠는지 적습니다.

**✏️ 회고**: 직접 만들어 보니 어땠는지 자유롭게 적습니다. 예를 들면 이런 것들입니다.
- 처음 세운 계획과 실제로 만들어진 구조가 어떻게 달랐는지
- Claude Code 에 다시 요청한 것과 그 까닭
- 오늘 미션의 코드와 다르게 만든 부분, 또는 미션 코드에서 가져다 쓴 부분

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

## 실습 미션 2: 내 질문을 평가셋에 더하고, 답변까지 채점해 네 칸으로 판단합니다
**내 질문 세 개를 평가셋에 더해 검색, 답변, 채점까지 실행하고, 정답 있는 문항을 네 칸으로 나눠 봅니다.**

재료는 `4일차_미션2_재료.txt`, 미션 1 의 색인과 검색기, 채점 프롬프트 `judge_prompt.txt` 입니다. 미션 1 에서는 검색만 측정했습니다. 미션 2 에서는 평가셋 10문항에 내 질문 세 개를 더해 검색 → 답변 → 채점까지 실행하고, 정답 있는 문항을 네 칸으로 나눕니다(예제 2-3, 2-5, 2-6).

| 칸 | 뜻 | 고칠 곳 |
|---|---|---|
| 찾고 맞음 | 정답 문서가 발췌에 있었고 답도 맞음 | 없음 |
| 찾고 틀림 | 정답 문서가 발췌에 있는데 답이 틀림 | 답변 생성(프롬프트, 발췌 개수, 청크) |
| 못 찾고 틀림 | 정답 문서가 발췌에 없어 답이 틀림 | 검색(검색기, 색인) |
| 못 찾고 맞음 | 정답 문서가 발췌에 없는데 답이 맞음 | 평가셋(정답 라벨) 점검 |

**✏️ 이 미션에서 손댈 곳**
- 직접 고칠 곳: 실습 미션 2 바꿔 보기 1 셀의 내 질문 `Q_A`, 바꿔 보기 2 셀의 발췌 수 `K`, 바꿔 보기 3 셀의 거절 규칙 `REFUSE_RULE`
- 더 해 보기(선택, 수업 뒤): 실습 미션 2 (a) 셀의 질문 세 개와 정답 문서번호
- 답 적는 곳: Q1, Q2
- 「문서 목록」, 「미션 2 실행 준비」 셀과 실습 미션 2 (a), (b) 셀은 실행만 합니다.

**하는 순서**
1. 「문서 목록」, 「미션 2 실행 준비」 셀과 실습 미션 2 (a), (b) 셀을 실행해 기본 결과를 봅니다. (a) 는 내 질문 세 개(예시 질문)를 확인하고, (b) 가 결과표 첫 줄(기준 줄)을 만듭니다.
2. **바꿔 보기 1, 2, 3**: 바꿔 보기 셀마다 ✏️ 줄 사이의 값 하나를 안내대로 고쳐 그 셀을 실행합니다. 바꿔 보기 셀은 그 셀의 값 하나만 기준 설정과 다르게 써서 검색 → 답변 → 채점을 실행하고, 결과표에 한 줄을 더해 기준 줄과 비교합니다. 바꿔 보기 1 에서 고친 질문은 그 셀 안에서만 쓰므로, 바꿔 보기 2, 3 의 줄은 (a) 의 질문으로 측정합니다.
3. 달라진 결과를 보고 미션 2 끝의 **Q1, Q2** 답 셀에 적습니다.
4. **✏️ 더 해 보기(선택, 수업 뒤)**: (a) 의 질문 세 개와 정답 문서번호를 내가 고른 것으로 바꿔 (a), (b) 를 한 번 더 실행합니다.

(b) 와 바꿔 보기 셀은 한 번 실행할 때 13문항에 문항마다 Claude 를 두 번(답변, 채점) 불러 1 ~ 2분 걸립니다. 같은 프롬프트는 저장해 둔 답을 다시 쓰므로, 바뀐 문항만 새로 호출합니다. LLM 채점이라 다른 사람이 실행하면 한두 문항의 판정이 다를 수 있습니다.

In [ ]:
# 문서 목록: 고치지 않고 실행합니다. 문서번호와 제목입니다
# 키(문서번호, 판)는 쓰지 않아 제목만 꺼냅니다
for title in TITLE.values():
    print(title[:60])                        # 제목 앞 60자만 출력합니다

**미션 2 실행 준비**: 실행만 합니다

이 셀은 도구를 만드는 셀이라 코드를 읽지 않아도 됩니다. 실행한 뒤 아래 표의 설정 세 개만 기억해 두세요.

예제 2-3 의 답변 프롬프트와 채점 프롬프트를 미션 코퍼스에 맞게 고치고, 결과표에 한 줄씩 더하는 `run2` 와 내 질문을 확인하는 `check_my` 를 정의합니다. 결과표는 (b) 셀이 만듭니다.

| 설정 | 뜻 | 기준 설정 값 |
|---|---|---|
| 발췌 | 답변 프롬프트에 붙이는 청크 수 | 4 |
| 검색기 | 발췌를 꺼낼 검색기 (미션 1 기준 설정의 리랭킹 검색기) | 'rerank' |
| 거절 규칙 | 「발췌에 없으면 근거를 찾지 못했다고만 답한다」 줄을 넣을지 | True |

결과표의 `있는데 거절` 은 문서에 답이 있는데 거절한 문항 수, `없는 질문 맞음` 은 문서에 답이 없는 질문을 바르게 거절한 문항 수입니다.

**출력에서 볼 것**: `미션 2 실행 준비 완료` 가 보이면 됩니다.

In [ ]:
# 미션 2 실행 준비: 고치지 않고 실행합니다
TEMPLATE = """너는 SL-PKG 조립기술 담당자다.
아래 사내 문서 발췌만 근거로 답한다.

[발췌]
{ctx}

규칙: 문장마다 근거 [문서번호]를 붙인다.
발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.

질문: {q}"""
REFUSAL = '사내 문서에서 근거를 찾지 못했습니다'
RULE_LINE = "발췌에 없으면 '사내 문서에서 근거를 찾지 못했습니다' 라고만 답한다.\n"
JUDGE = open('judge_prompt.txt', encoding='utf-8').read()
# 미션 1 의 기준 설정 검색기
RANK = {'rerank': rerank_rank, 'vector': vec_rank, 'keyword': bm_rank}

def format_ctx(hits):
    """청크 목록을 받아 [문서번호_판] (절) 과 본문을 이어 붙인 발췌 문자열을 돌려줍니다."""
    return '\n\n'.join(f'[{h.doc_id}_{h.version}] ({h.section})\n{h.text}' for h in hits)

def retrieve(q, mode, k):
    """질문 · 검색기 이름 · 발췌 수를 받아 그 검색기의 상위 k개 청크 목록을 돌려줍니다."""
    return [chunks[i] for i in RANK[mode](q)[:k]]

def judge(e, ans, hits):
    """평가 문항 · 답변 · 발췌 청크를 받아 충실도 · 근거일치 · 정답성 판정 dict 를 돌려줍니다 (예제 2-3 (3) 과 같은 채점)."""
    # 정답 본문은 앞 1800자까지만 채점 프롬프트에 넣습니다
    gold = '\n'.join(c.text for c in DOCT[e['gold']])[:1800] if e['gold'] in DOCT else '(정답 문서 없음)'
    # 발췌는 앞 3000자까지, 채점 답 길이 상한 1000
    out = llm(JUDGE.format(q=e['q'], ctx=format_ctx(hits)[:3000], ans=ans, gold=gold), 1000)
    def pick(key):
        match = re.search(key + r'\s*:\s*(예|아니오|부분)', out)
        return match.group(1) if match else '?'
    return {'faith': pick('충실도'), 'cite': pick('근거일치'), 'correct': pick('정답성')}

def box_of(r):
    """문항 결과 dict 를 받아 발췌 적중 × 정답 여부로 네 칸 가운데 하나의 칸 이름을 돌려줍니다."""
    return ('찾고' if r['retrieved'] else '못 찾고') + (' 맞음' if r['correct'] == '예' else ' 틀림')

def check_my(my=None):
    """정답 있는 질문은 질문 단어가 정답 문서에 있는지, 문서에 없는 질문은 문서 모음 어디에도 없는 단어를 보입니다."""
    # my 를 주면 그 질문 목록을 확인합니다(바꿔 보기 1)
    for e in (MY if my is None else my):
        # 한 글자 단어는 뺍니다
        question_words = [w for w in dict.fromkeys(re.sub(r'[^\w가-힣-]+', ' ', e['q']).split()) if len(w) > 1]
        print('\n질문 ·', e['q'])
        if not e['gold']:
            all_text = '\n'.join(c.text for c in chunks)
            print('   문서에 없는 질문 · 질문 단어 중 문서 모음 어디에도 없는 것', [w for w in question_words if w not in all_text])
            continue
        if e['gold'] not in DOCT:
            print('   문서번호', e['gold'], '가 문서 모음에 없습니다 · 위 문서 목록에서 다시 골라 주세요')
            continue
        if e['gold'] in {x['gold'] for x in EV}:
            print('   [안내]', e['gold'], '는 평가셋 10문항의 정답 문서입니다 · 평가셋에 없는 문서로 골라 주세요')
        body = doc_text(e['gold'])
        hit = [w for w in question_words if w in body]
        print('   정답 문서', e['gold'], '· 질문 단어 중 이 문서에 있는 것', hit)
        # 정답 문서에 있는 질문 단어를 모두 가진 다른 문서가 있으면 정답이 하나로 정해지지 않습니다
        others = [d for d in DOCT if d != e['gold'] and all(w in doc_text(d) for w in hit)]
        # 6건까지만 보이고 더 많으면 건수만 붙입니다
        print('   이 단어를 모두 가진 다른 문서', others[:6], f'(모두 {len(others)}건)' if len(others) > 6 else '')

# 기준 설정: (b) 셀과, 바꿔 보기 셀이 고치지 않는 값은 이 값을 씁니다
BASE2 = dict(k=4, searcher='rerank', rule=True)
NAME2 = {'k': '발췌', 'searcher': '검색기', 'rule': '거절 규칙'}
LOG2, ROWS2 = [], []
def run2(k=4, searcher='rerank', rule=True, my=None):
    """세 값으로 평가셋 10문항 + 내 질문을 검색 → 답변 → 채점하고, 네 칸 결과표와 기준 대비 칸이 바뀐 문항을 보입니다."""
    settings = dict(k=k, searcher=searcher, rule=rule)
    for key, value in settings.items():
        if type(value) is not type(BASE2[key]):
            return print(f"[안내] {NAME2[key]} 에 {value!r} 는 넣을 수 없습니다 · 처음 값처럼 넣어 주세요 (예: 8 · 'vector' · False)")
    if searcher not in RANK:
        return print("[안내] 검색기는 'rerank' · 'vector' · 'keyword' 가운데 하나를 따옴표까지 넣어 주세요")
    # my 를 주면 그 질문 목록으로 측정합니다(바꿔 보기 1)
    QS = EV + (MY if my is None else my)
    changed = [f'{NAME2[key]} {settings[key]}' for key in settings if settings[key] != BASE2[key]]
    if ROWS2 and [r['q'] for r in ROWS2[0]] != [e['q'] for e in QS]:
        base_q = [r['q'] for r in ROWS2[0][len(EV):]]
        which = [name for name, old_q, new in zip(['Q_A', 'Q_B', 'Q_C'], base_q, QS[len(EV):]) if old_q != new['q']]
        # 바뀐 질문 이름을 붙입니다. 이름이 하나도 다르지 않으면 질문 개수가 달라진 것이라 '수' 를 붙입니다
        changed.append(f"내 질문 바꿈({' · '.join(which) or '수'})")
    # 기준 줄이 먼저 있어야 비교할 수 있어 기준부터 측정합니다
    if not LOG2 and changed:
        run2()
    if LOG2 and not changed:
        print('바꿀 값과 내 질문이 기준과 같아 줄을 더하지 않았습니다 · 지금까지의 결과표입니다')
        return display(pd.DataFrame(LOG2).set_index('실행'))
    if len(changed) > 1:
        print('[안내] 값 두 개 이상을 함께 바꿨습니다 · 이 줄만으로는 어느 값 때문인지 구분할 수 없습니다')
    label = ' · '.join(changed) or '기준'
    same = sum(r['실행'].split(' (')[0] == label for r in LOG2)
    # 같은 설정을 다시 측정하면 몇 번째인지 붙입니다
    if same:
        label += f' ({same + 1}번째)'
    prompt = TEMPLATE if settings['rule'] else TEMPLATE.replace(RULE_LINE, '')
    # 검색은 차례로
    prepared = [retrieve(e['q'], settings['searcher'], settings['k']) for e in QS]
    def one(e, hits):
        ans = llm(prompt.format(ctx=format_ctx(hits), q=e['q']))
        return {'q': e['q'], 'gold': e['gold'], 'retrieved': e['gold'] in {h.doc_id for h in hits},
                'top': [h.doc_id for h in hits], 'refused': ans.startswith(REFUSAL), 'ans': ans, **judge(e, ans, hits)}
    # 답변, 채점만 최대 4개씩 동시에 호출합니다
    with ThreadPoolExecutor(4) as ex:
        rows = list(ex.map(one, QS, prepared))
    box = Counter(box_of(r) for r in rows if r['gold'])
    row = {'실행': label, '발췌': settings['k'], '검색기': settings['searcher'], '거절 규칙': settings['rule'],
           **{b: box[b] for b in ['찾고 맞음', '찾고 틀림', '못 찾고 틀림', '못 찾고 맞음']},
           '있는데 거절': sum(r['refused'] for r in rows if r['gold']),
           '없는 질문 맞음': f"{sum(r['correct'] == '예' for r in rows if not r['gold'])} / {sum(not r['gold'] for r in rows)}"}
    LOG2.append(row)
    ROWS2.append(rows)
    print('결과표 · 정답 있는 문항을 네 칸(정답 문서가 발췌에 있었나 × 답이 맞았나)으로 집계합니다 · 맨 위 줄이 기준입니다')
    display(pd.DataFrame(LOG2).set_index('실행'))
    print('\n내 질문 세 개의 결과 · 발췌 문서번호 · 칸 · 답 앞 90자')
    for r in rows[len(EV):]:
        kind = box_of(r) if r['gold'] else ('없는 질문 맞음' if r['correct'] == '예' else '없는 질문 틀림')
        print(f"   {r['q'][:40]}\n     정답 {r['gold'] or '없음'} · 발췌 {r['top']} · {kind}")
        print('     답 │', r['ans'][:90].replace('\n', ' '))
    if len(ROWS2) > 1:
        print('\n기준 줄과 비교해 칸 또는 거절 여부가 바뀐 평가셋 문항 · 답은 앞 90자')
        moved = 0
        for b, r in zip(ROWS2[0][:len(EV)], rows[:len(EV)]):
            if box_of(b) != box_of(r) or b['refused'] != r['refused']:
                moved += 1
                print(f"\n   {r['gold']} · {box_of(b)} → {box_of(r)} · 거절 {b['refused']} → {r['refused']}   질문: {r['q'][:40]}")
                print('     기준 답 │', b['ans'][:90].replace('\n', ' '))
                print('     이번 답 │', r['ans'][:90].replace('\n', ' '))
        if not moved:
            print('   없음')

print('미션 2 실행 준비 완료 · (a) 셀에서 내 질문을 확인하고 (b) 셀에서 결과표를 만듭니다')

In [ ]:
# 실습 미션 2 (a) 평가셋 10문항에 더할 내 질문 세 개를 적고 확인합니다 (예제 2-1, 2-2 질문 만들기)
# 볼 것: 질문 단어가 정답 문서에 있는지, 그 단어를 모두 가진 다른 문서가 있는지(있으면 정답이 하나로 안 정해짐)

# ✏️ ======== 더 해 보기(선택, 수업 뒤): 평가셋에 더할 내 질문 세 개와 정답 문서번호 ========
# 지금은 예시 질문이 들어 있습니다. 질문 1 은 호기와 레시피 번호로, 질문 2 는 날짜와 호기로 정답 문서를 하나로 좁혔고, 질문 3 은 문서에 없는 레시피를 묻습니다.
# 기본 미션을 마친 뒤 「문서 목록」 셀에서 고른 문서로 질문과 정답 문서번호를 고쳐 (a), (b) 를 다시 실행하면, 내 질문이 어느 칸에 드는지 확인할 수 있습니다.
Q_A = 'MOLD-01 호기 RCP-MOLD-445 레시피의 금형 온도 관리 한계는?'
GOLD_A = 'SPEC-PKG-011'
Q_B = '2025-02-27 DB-01 호기 몰드 보이드 건의 영향 스트립 수량은?'
GOLD_B = 'QIR-PKG-2025-0397'
Q_C = 'WB-09 호기 RCP-WB-999 레시피의 본딩 하중 목표값은?'
# ✏️ ======== 여기까지 ========

MY = [{'q': Q_A, 'gold': GOLD_A}, {'q': Q_B, 'gold': GOLD_B}, {'q': Q_C, 'gold': ''}]
check_my()

In [ ]:
# 실습 미션 2 (b) 평가셋 10문항 + 내 질문 3문항을 검색 → 답변 → 채점합니다 (예제 2-3, 2-5, 2-6)
# 볼 것: 결과표의 네 칸, 있는데 거절, 없는 질문 맞음, 내 질문 세 개의 칸과 답. 이 줄이 기준 줄입니다

# 기준 설정: 발췌 4, 검색기 'rerank', 거절 규칙 True
run2(**BASE2)


### 미션 2: 바꿔 보고 답하기
**바꿔 보기 셀마다 ✏️ 줄 사이의 값 하나를 안내대로 고쳐 실행합니다. 바꿔 보기 셀은 그 셀의 값 하나만 기준 설정과 다르게 쓰므로, 다른 바꿔 보기의 결과와 섞이지 않습니다.**

**✏️ 바꿔 보기 1** 실습 미션 2 바꿔 보기 1 셀의 `Q_A` 에서 호기와 레시피 번호를 지워 `Q_A = '금형 온도 관리 한계는?'` 으로 고치고 실행합니다. 「이 단어를 모두 가진 다른 문서」 줄과, 내 질문 결과에서 이 질문의 발췌 문서번호, 칸, 답을 봅니다. 고친 질문은 이 셀에서만 쓰고, (a) 셀의 질문은 그대로입니다.

**✏️ 바꿔 보기 2** 바꿔 보기 2 셀에서 답변 프롬프트에 붙일 발췌 수 `K = 4` 를 `8` 로 고쳐 실행합니다. 결과표의 `발췌 8` 줄에서 네 칸(`찾고 맞음`, `찾고 틀림`, `못 찾고 틀림`, `못 찾고 맞음`)과 `있는데 거절` 을 기준 줄과 비교합니다. 결과표 아래 「칸 또는 거절 여부가 바뀐 평가셋 문항」 목록도 봅니다.

**✏️ 바꿔 보기 3** 바꿔 보기 3 셀에서 답변 프롬프트에 넣을 거절 규칙 `REFUSE_RULE = True` 를 `False` 로 고쳐 실행합니다. 결과표의 `거절 규칙 False` 줄에서 `있는데 거절`, `없는 질문 맞음` 과 네 칸을 기준 줄, `발췌 8` 줄과 비교합니다. 내 질문 Q_B, Q_C 의 답은 (b) 셀의 출력과 비교합니다.

**✏️ 더 해 보기(선택, 수업 뒤)** (a) 의 세 질문을 내가 고른 문서에서 만든 질문으로 바꿔 (a), (b) 를 한 번 더 실행합니다. 정답 문서는 「문서 목록」 셀에서 평가셋 10문항의 정답 문서가 아닌 것으로 고릅니다.

In [ ]:
# 실습 미션 2 바꿔 보기 1: 내 질문 Q_A 에서 호기와 레시피 번호를 지운 질문으로 확인하고 채점합니다
# 볼 것: 「이 단어를 모두 가진 다른 문서」 줄, 내 질문 결과에서 이 질문의 발췌 문서번호, 칸, 답

# ✏️ ======== 직접 고칠 곳: 호기와 레시피 번호를 지울 내 질문 Q_A ========
# 지금은 실습 미션 2 (a) 의 질문 1 과 같은 질문이 들어 있습니다.
# 호기와 레시피 번호를 지워 '금형 온도 관리 한계는?' 으로 고쳐, 식별자가 없으면 정답 문서가 하나로 정해지는지와 답이 어떻게 바뀌는지 확인합니다.
Q_A = 'MOLD-01 호기 RCP-MOLD-445 레시피의 금형 온도 관리 한계는?'
# ✏️ ======== 여기까지 ========

# 고친 Q_A 는 이 셀에서만 씁니다. (a) 의 MY 는 그대로입니다
my_questions = [{'q': Q_A, 'gold': MY[0]['gold']}] + MY[1:]
# 고친 Q_A 한 문항만 확인합니다. 나머지 둘은 (a) 와 같습니다
check_my(my_questions[:1])
run2(**BASE2, my=my_questions)                   # 질문 말고 세 값은 기준 설정 그대로


In [ ]:
# 실습 미션 2 바꿔 보기 2: 답변 프롬프트에 붙일 발췌 수를 늘려 채점합니다
# 볼 것: 결과표의 네 칸과 있는데 거절, 기준 줄과 비교해 칸이 바뀐 문항

# ✏️ ======== 직접 고칠 곳: 답변 프롬프트에 붙일 발췌 수 ========
# 지금은 4 가 들어 있어 검색 상위 발췌(청크) 4개를 답변 프롬프트에 붙입니다.
# 8 로 고쳐, 발췌를 늘리면 네 칸과 있는데 거절 문항 수가 어떻게 바뀌는지 확인합니다.
K = 4
# ✏️ ======== 여기까지 ========

# 나머지 두 값은 기준 설정 그대로
run2(k=K, searcher=BASE2['searcher'], rule=BASE2['rule'])


In [ ]:
# 실습 미션 2 바꿔 보기 3: 답변 프롬프트에서 거절 규칙을 빼고 채점합니다
# 볼 것: 있는데 거절, 없는 질문 맞음, 네 칸, 내 질문 Q_B, Q_C 의 답

# ✏️ ======== 직접 고칠 곳: 답변 프롬프트에 넣을 거절 규칙 ========
# 지금은 True 가 들어 있어 답변 프롬프트에 「발췌에 없으면 근거를 찾지 못했다고만 답한다」 규칙이 들어갑니다.
# False 로 고쳐, 이 규칙을 빼면 있는데 거절, 없는 질문 맞음과 내 질문 Q_B, Q_C 의 답이 어떻게 바뀌는지 확인합니다.
REFUSE_RULE = True
# ✏️ ======== 여기까지 ========

# 나머지 두 값은 기준 설정 그대로
run2(k=BASE2['k'], searcher=BASE2['searcher'], rule=REFUSE_RULE)


**Q1: 질문에서 호기와 레시피 번호를 지우면 어떻게 되나요?**

- **의도**: 평가셋 질문에 식별자(호기, 레시피 번호)가 없으면 정답 문서가 하나로 정해지지 않는다는 것을 확인합니다.
- **입력**: 바꿔 보기 1 셀에서 `Q_A` 를 `'금형 온도 관리 한계는?'` 으로 고쳐 실행합니다. (a) 셀은 고치지 않습니다.
- **기대 결과**: 출력 위쪽 「이 단어를 모두 가진 다른 문서」 줄에 다른 문서가 나옵니다. 여섯 건이 넘으면 줄 끝 괄호에 전체 건수가 붙습니다. 결과표에 `내 질문 바꿈(Q_A)` 줄이 생기고, 아래 「내 질문 세 개의 결과」에 이 질문의 발췌 문서번호, 칸, 답이 나옵니다.
- **답에 적을 것**: (1) 「이 단어를 모두 가진 다른 문서」 건수 (2) 이 질문의 발췌 문서번호, 칸, 답 (3) 평가셋에 질문을 더할 때 식별자를 넣어 정답 문서를 하나로 좁혀야 하는 까닭 한 문장
- **답 예시(모양만)**: 다른 문서 ○○건 / 발췌 문서번호 ○○○, ○○○ / ○○ 칸 / 답은 ○○○ / 식별자가 없으면 ○○○ 이라서 채점 결과를 믿을 수 없습니다.

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

**Q2: 발췌를 늘리거나 거절 규칙을 빼면 거절이 줄어드나요? 맞은 답도 늘어나나요?**

- **의도**: `있는데 거절` 이 줄어도 맞은 답이 늘었는지는 따로 확인해야 한다는 것을 봅니다.
- **입력**: 바꿔 보기 2 셀에서 `K = 4` 를 `8` 로, 바꿔 보기 3 셀에서 `REFUSE_RULE = True` 를 `False` 로 고쳐 각각 실행합니다.
- **기대 결과**: 결과표에 `발췌 8` 줄과 `거절 규칙 False` 줄이 생깁니다. 두 줄의 `있는데 거절` 과 `찾고 맞음` 칸을 기준 줄과 비교합니다.
- **답에 적을 것**: (1) 두 줄 각각 `있는데 거절` 과 `찾고 맞음` 이 기준 줄에서 얼마로 바뀌었는지 (2) 바꿔 보기 3 에서 날짜가 든 내 질문 Q_B 의 답이 어떻게 바뀌었는지 (3) `있는데 거절` 을 줄이려면 답변 프롬프트, 발췌 수, 청크(색인) 가운데 어디를 고칠지와 근거 한 문장
- **답 예시(모양만)**: 발췌 8: 있는데 거절 ○ → ○, 찾고 맞음 ○ → ○ / 거절 규칙 False: 있는데 거절 ○ → ○, 찾고 맞음 ○ → ○ / Q_B 답은 ○○○ / ○○ 를 고칩니다. ○○○ 때문입니다.

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

### 확장 미션 2 (심화): 답변 채점 도구 만들기
**기본 미션을 마친 경우에만 합니다. 목적만 보고 어떻게 만들지 스스로 정한 뒤, Claude Code 로 직접 만들어 봅니다.**

**목적**
사내 문서 질의응답 도구를 고칠 때마다 답변 품질을 같은 기준으로 측정해야 합니다. 질문 목록을 넣으면 검색, 답변, 채점까지 한 번에 실행하고, 문항마다 정답 문서를 찾았는지와 답이 맞았는지를 나눠 보여 주는 도구가 필요합니다.

**재료**: `mission_corpus/` 폴더, `mission_manifest.csv`, `mission_evalset.csv`, `judge_prompt.txt`. 학습 사이트 4일차의 `day04v2-missions.zip` 을 내 PC 에 받아 풀면 들어 있습니다(코랩에 풀린 폴더는 Claude Code 가 열 수 없습니다). 본인 업무에 맞게 재료를 새로 만들어 진행해도 됩니다.

**만들기 전에 생각해 볼 것**
- 답이 틀린 문항을 검색 문제와 답변 문제로 어떻게 나눌까요?
- 모델에게 채점을 맡길 때, 같은 문항의 판정이 실행마다 달라지면 어떻게 다룰까요?
- 문서에 답이 없는 질문에는 어떤 답을 맞았다고 볼까요?

**진행**
1. 목적을 읽고, 어떤 단계로 만들지 먼저 짧게 메모합니다.
2. PC 에 작업 폴더를 만들고 재료 파일을 넣은 뒤, Claude 앱 Code 탭에서 그 폴더를 열어 **새 세션**을 시작합니다.
3. 목적과 내 계획을 Claude Code 에 보내 만들게 하고, 결과를 보며 고쳐 달라고 요청하면서 완성해 갑니다.
4. 다 만들면 바로 아래 「회고」에 직접 만들어 보니 어땠는지 적습니다.

**✏️ 회고**: 직접 만들어 보니 어땠는지 자유롭게 적습니다. 예를 들면 이런 것들입니다.
- 처음 세운 계획과 실제로 만들어진 구조가 어떻게 달랐는지
- Claude Code 에 다시 요청한 것과 그 까닭
- 오늘 미션의 코드와 다르게 만든 부분, 또는 미션 코드에서 가져다 쓴 부분

**답**: ✏️ 답 적는 곳 (두 번 클릭해 적기)

(여기에 적습니다)

## 실습 미션 3: Claude Code 로 위키를 갱신합니다 (재료 `day04-wiki/`)
**미션 3 은 Claude 앱 Code 탭에서 합니다. 이 노트북에서는 실행할 셀이 없습니다. 예제 3-1, 3-2 에서 쓴 `day04-wiki/` 폴더를 그대로 열고 이어서 합니다.**

1. 새 문서 넣기: `day04-wiki/새자료/QIR-2026-0931.txt` 를 `원본/` 폴더로 옮깁니다(잘라내 붙이기).
2. 갱신: `day04-wiki/prompts/4_위키갱신.txt` 를 열어 전부 복사해 Code 탭 새 세션 입력창에 붙여 넣고 보냅니다. Claude Code 가 새 문서를 읽고, 예제에서 만든 위키(`wiki/`)의 관련 페이지와 `index.md`, `log.md` 를 고칩니다. 갱신 중 명령 실행 권한 카드가 뜨면 명령을 읽어 봅니다. cat, ls, grep, diff 처럼 읽거나 비교만 하는 명령이면 「한 번만 허용」을 누르고, day04-wiki 폴더 밖(임시 폴더 등)에 쓰거나 `원본/` 을 고치거나 지우는 명령이면 허용하지 않습니다.
3. 바뀐 곳과 새로 생긴 `[검토 필요]` 를 원본 문서와 대조해 점검합니다.
4. 바꿔 보기 둘과 Q1, Q2 는 `4일차_미션3_재료.txt` 에 있습니다. 답은 `4일차_미션3_답안.txt` 에 적습니다.

## 마치며
**미션 1, 2 를 마친 노트북과 미션 3 의 두 파일을 zip 하나로 묶어 휴넷 랩스에 올립니다.**

**제출하는 법**
1. 미션 1 의 Q1 ~ Q3, 미션 2 의 Q1, Q2 답 셀을 모두 적었는지 확인합니다. 확장 미션을 했다면 「회고」 아래 답 셀도 확인합니다.
2. [파일] → [다운로드] → [.ipynb 다운로드] 를 눌러, 실행 결과가 남은 노트북을 내려받습니다.
3. 파일 이름을 `아이디_4일차_실습미션.ipynb` 로 바꿉니다. 예: `aaa_4일차_실습미션.ipynb`
4. 이 노트북과 미션 3 의 `4일차_미션3_답안.txt`, `wiki/log.md` 를 zip 하나로 묶어 [휴넷 랩스](https://dssw.labs.hunet.com/)에 올립니다. 휴넷 랩스에는 .ipynb 파일이 바로 올라가지 않습니다.